# LLaMA-Factory 框架
LLaMA-Factory是工业界最主流、使用最广泛的开源微调框架。我们将使用该框架完成一次全流程的 SFT 指令微调，并掌握关键训练超参数调优与 Loss 曲线分析。

## LLaMA-Factory 极速微调与训练监控
1. LLaMA-Factory 核心架构：理解其集成的 Dataset/Model/Trainer 一站式解耦设计。
2. SFT 核心训练超参数：深刻理解 `learning_rate`（学习率）、`warmup_ratio`(预热比例)、`gradient_accumulation_steps`（梯度累加步数）与 `lr_scheduler_type` 的配合逻辑。
3. 编写 CLI 配置文件运行 QLoRA 微调，并通过 Loss 曲线分析拟合状态。

## LLaMA-Factory 运行机制与核心超参数解析
LLaMA-Factory 是目前大模型微调领域功能最全、效率极高的开箱即用开源框架，它将复杂的 Hugging Face `Trainer`、`PEFT`、`DeepSpeed` 以及 `bitsandbytes` 封装为简单易用的 CLI 命令或 YAML 配置文件。

1. 核心训练超参数解析
    * `per_device_train_batch_size` (单卡 Micro Batch Size)：单张 GPU 在每次前向传播时同时处理的样本数。由于显存限制，在单卡 16G/24G 上通常设为 `1` 或 `2`。
    * `gradient_accumulation_steps` (梯度累加步数)：为了实现更大的全局 Batch Size（Global Batch Size），我们可以通过连续累加 $N$ 次前向/反向传播的梯度后再执行一次优化器更新（`optimizer.step()`）：

        $$\text{Global Batch Size} = \text{per\_device\_train\_batch\_size} \times \text{gradient\_accumulation\_steps} \times \text{GPU\_Count}$$

    * `learning_rate` (学习率)：
        * 全量微调（Full FT）：通常较小，如 $1 \times 10^{-5} \sim 2 \times 10^{-5}$。
        * LoRA / QLoRA 微调：因为只更新少量参数，通常需要较大的学习率，如 $1 \times 10^{-4} \sim 2 \times 10^{-4}$。

    * `warmup_ratio` (预热比例)：在训练前 5%~10% 的 Step 中，学习率从 0 线性增加到设定的 `learning_rate`，防止训练刚开始时未初始化的梯度冲击模型原有权重。

#### 使用 LLaMA-Factory YAML 配置运行 QLoRA
> 在终端运行：
`git clone --depth 1 [https://github.com/hiyouga/LLaMA-Factory.git](https://github.com/hiyouga/LLaMA-Factory.git)
cd LLaMA-Factory && pip install -e .[metrics]`

1. 步骤 1：注册自定义数据集 (`dataset_info.json`)
    在 `LLaMA-Factory/data/dataset_info.json` 中追加你自定义的数据集配置：

        "my_custom_agent_sft": {
          "file_name": "my_agent_data.json",
          "formatting": "sharegpt",
          "columns": {
            "messages": "messages"
          }
        }

2. 步骤 2：编写 QLoRA 微调配置文件 (qwen2_5_qlora_sft.yaml)

        ### model
        model_name_or_path: Qwen/Qwen2.5-7B-Instruct

        ### method
        stage: sft
        do_train: true
        finetuning_type: lora
        quantization_bit: 4            # 启用 4-bit QLoRA 训练
        quantization_method: bitsandbytes

        ### dataset
        dataset: my_custom_agent_sft
        template: qwen                 # 必须严格指定对应的对话 Chat Template
        cutoff_len: 2048               # 最大序列长度
        overwrite_cache: true
        preprocessing_num_workers: 8

        ### output
        output_dir: saves/qwen2.5-7b/qlora/sft
        logging_steps: 10
        save_steps: 100
        plot_loss: true                # 自动绘制 Loss 训练曲线
        overwrite_output_dir: true

        ### train
        per_device_train_batch_size: 2
        gradient_accumulation_steps: 4 # Global Batch Size = 2 * 4 = 8
        learning_rate: 2.0e-4          # LoRA / QLoRA 推荐学习率
        num_train_epochs: 3.0
        lr_scheduler_type: cosine      # 使用余弦退火学习率调度器
        warmup_ratio: 0.1
        fp16: false
        bf16: true                     # 推荐在 Ampere 架构（RTX 3090/4090）开启 bf16

        ### lora
        lora_rank: 16
        lora_alpha: 32
        lora_dropout: 0.05
        lora_target: all               # 自动将 LoRA 挂载到所有 Linear 层 (Q,K,V,O, Gate, Up, Down)

3. 步骤 3：启动一键训练命令:
        `llamafactory-cli train qwen2_5_qlora_sft.yaml`

#### 训练 Loss 曲线诊断与拟合判断
训练开始后，LLaMA-Factory 会在 `saves/qwen2.5-7b/qlora/sft` 目录下生成 `training_loss.png` 图片。我们需要学会判断训练状态：

1. 正常收敛状态：
    * 训练刚开始时 Loss 在 2.0~3.0 左右，在前 20% 的 Steps 中迅速下降，最终平缓收敛在 0.5 ~ 0.8 之间。

2. 过拟合（Overfitting）：
    * Loss 被刷到了 0.1 甚至 0.0x 以下。模型失去了通用推理能力，变为了“死记硬背”训练集的检索数据库。表现为测试集泛化极差，只对训练集里的问答吐字正常。
    * 解决方法：调大 `lora_dropout`（如 0.1）、降低 `learning_rate`、减少 `num_train_epochs` 或补充更多多样化数据。

3. 欠拟合（Underfitting） / 梯度爆炸：
    * Loss 一直在 2.0 高位振荡不下降，或者瞬间变成 `NaN`。
    * 解决方法：检查是否未对 Prompt 做 Mask、降低学习率、将 `fp16` 切换为数值稳定性更高的 `bf16`。

**思考**

1. 多卡分布式微调（Multi-GPU）：如果手头有 4 张 RTX 4090 GPU，在 LLaMA-Factory 中使用 PyTorch Distributed Data Parallel (DDP) 或 DeepSpeed Stage 2/3 进行并行训练时，启动命令应该有什么变化？
2. Eval 验证集配置：为了防止模型在 SFT 过程中过拟合，如何在配置 YAML 中划分 10% 的验证集（Validation Set），并开启 `eval_steps` 和 `early_stopping`？

## 微调全栈复盘与 LoRA 融合导出
1. 微调全栈技术选型：明确 Full FT、LoRA 与 QLoRA 在不同资源条件下的决策路径。
2. LoRA 权重的物理合并与格式导出：学会将训练好的 LoRA 适配器融合回 Base Model，并导出为 FP16 Hugging Face 格式或 GGUF 格式。
3. 通过硬核场景考题，检验你对微调原理、显存预算、数据 Masking 及工程落地的掌握程度。

#### 核心知识体系全景复盘

|技术方案|参数更新比例|显存需求 (7B 模型)|优点|缺点 / 局限| 最佳适用场景   |
|----|----|----|----|----|----------|
|全量微调 (Full FT)|100%|$\approx 80\text{GB} \sim 110\text{GB}$|表现上限高，对底层领域知识的重构能力强。|显存消耗极大（需要 A100/H100 集群），容易产生灾难性遗忘。|基础大模型二次预训练、极度垂直且庞大的专属领域基座构建。|
|LoRA (BF16/FP16)|$\approx 0.05\% \sim 0.2\%$|$\approx 14\text{GB} \sim 18\text{GB}$|训练稳定，可叠加多个适配器，无额外推理延迟。|依然需要全量高精度基座显存开销。|拥有单卡 24G/40G/80G 显卡，追求极高训练效果与稳定的场景。|
|QLoRA (NF4 4-bit)|$\approx 0.05\% \sim 0.2\%$|$\approx 5\text{GB} \sim 8\text{GB}$|极致省显存，消费级显卡（RTX 3090/4090/4060Ti）即可训练 7B~14B 模型。|4-bit 权重反量化前向计算略有一点点吞吐开销。|算力受限、私有化部署前快速试错、个人开发者与中小企业。|

#### LoRA 适配器权重物理合并（Merge and Unload）
在生产环境中，如果我们每次推理都要同时加载 Base Model 和 LoRA 适配器，会增加不必要的内存搬运开销。最佳实践是在微调完成后，将 LoRA 的 $B \cdot A$ 权重物理相加融合回原始权重矩阵 $W_0$，生成一个新的独立模型权重。

**使用 `peft` 执行权重合并导出:**

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_PATH = "Qwen/Qwen2.5-7B-Instruct"
LORA_PATH = "saves/qwen2.5-7b/qlora/sft"
EXPORT_PATH = "models/qwen2.5-7b-custom-agent-merged"

print("🚀 正在加载原始 Base Model (FP16/BF16) ...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="cpu",  # 合并通常可以在 CPU 内存上完成 (建议 32G+ 内存)
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)

print("🚀 正在加载 LoRA 适配器 ...")
lora_model = PeftModel.from_pretrained(base_model, LORA_PATH)

print("🔗 正在执行物理权重融合 (W_merged = W_0 + B * A) ...")
# merge_and_unload 会计算并将 LoRA 权重融入基座，返回纯粹的 PreTrainedModel 对象
merged_model = lora_model.merge_and_unload()

print(f"💾 正在保存合并后的全量模型至: {EXPORT_PATH} ...")
merged_model.save_pretrained(EXPORT_PATH, safe_serialization=True)
tokenizer.save_pretrained(EXPORT_PATH)

print("✅ 权重合并与导出成功！导出后的模型可直接输入 vLLM、Ollama 或转换为 GGUF 格式部署！")

1. 考核题 1（微调原理与 Loss 机制）：
    * 在构建 Agent 工具调用（Tool Calling）的 SFT 数据集时，数据包含了 `[System, User, Thought, Action, Observation, Final Response]`。
    * 请问：在计算 Cross-Entropy Loss 时，除了 `System` 和 `User` 的 Prompt 必须被 Mask 为 `-100` 之外，工具调用的执行结果 `Observation`（由环境或外部 API 返回）是否应该被 Mask 为 `-100`？为什么？

2. 考核题 2（显存计算与 QLoRA 细节）：
    * 假设你正在使用 QLoRA 训练一个 14B 参数的模型。基座模型使用 4-bit NF4 量化，LoRA 挂载在所有的 Linear 层（$r=16$），`max_seq_length` 设定为 4096。
    * 如果在训练过程中突然遭遇 OOM（Out of Memory）异常，但在 Batch Size 已降为 1 的情况下依然发生 OOM，作为架构师，你应该从哪三个维度（例如：梯度累加、梯度检查点 Gradient Checkpointing、序列截断、分页优化器）来进一步削减显存？

3. 考核题 3（微调与 RAG/Agent 架构选型）：
    * 业务部门提出要求：“希望能让 AI 系统准确回答公司最新的 500 页内部员工手册和报销制度”。
    * 项目经理建议：“我们拿这 500 页文档对 Qwen2.5 进行一次 SFT 微调，让大模型把制度记在脑子里”。
    * 请从“知识更新成本、幻觉可控性、事实精准度”三个角度分析，项目经理的微调方案是否合适？更好的架构方案应该是什么？